# Nonconvex MM tomography: Table 4

Run the cells in order. Figures, tables, and their data are saved in `results/`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "src" / "tomography"))
sys.path.insert(0, str(ROOT / "src" / "tomography" / "vendor"))

from compute_backend import DEVICE
print(f"Device: {DEVICE.upper()} (tomography preconditioner; projection and outer solver use CPU)")

from experiment_2d import (save_solver_result, save_raw_results,
                           save_image_data, save_curve_data, save_table_png)
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


import scipy.sparse as sps
from scipy.sparse.linalg import cg as sp_cg

import pandas as pd

from trips.test_problems.Tomography import Tomography
import astra

import jlinops

import numpy as np
import math

from jlinops.base import _CustomLinearOperator
# from .util import get_device

# from ias import IASSolver

from trips.utilities.phantoms import shepp_logan

from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import FormatStrFormatter

import matplotlib as mpl
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.gridspec as gridspec
# mpl.rcParams['text.usetex'] = True
# mpl.rc('text.latex', preamble=r'\usepackage{amsmath}')

# Define the custom style with RGB color values
custom_style = {
    "axes.prop_cycle": plt.cycler(color=[(106/255, 192/255, 205/255),  # blue - C0
                                         (235/255, 186/255, 63/255),  # gold - C1
                                         (172/255, 46/255, 55/255),  # red - C2
                                         (32/255, 82/255, 145/255),  # dark blue - C3
                                         (83/255, 165/255, 106/255),  # green - C4
                                         (239/255, 145/255, 75/255),  # orange - C5
                                         (208/255, 92/255, 89/255),  # pink - C6
                                         (80/255, 67/255, 157/255),  # purple - C7
                                         (82/255, 82/255, 82/255),  # gray - C8
                                         (53/255, 53/255, 53/255),  # charcoal - C9
                                         (30/255, 27/255, 28/255), # black - C10  
                                         ]) 
}
plt.rc('axes', prop_cycle=custom_style['axes.prop_cycle'])

method_markers = {
    "GKS": "o",
    "S-GKS": "o",
    "resS-GKS": "v",
    "recS-GKS": "^",
    "PS-GKS": "o",
    "resPS-GKS": "v",
    "recPS-GKS": "^",
    "PS-GKB": "o",
    "FGK": "o",
    "FLSQR-I": "$\\text{I}$",
    "FLSQR-R": "$\\text{R}$",
    "FLSQR-W": "$*$",
    "IRW-FLSQR": "x",
}

method_colors = {
    "GKS": "C1",
    "S-GKS": "C4",
    "resS-GKS": "C4",
    "recS-GKS": "C4",
    "PS-GKS": "C2",
    "resPS-GKS": "C2",
    "recPS-GKS": "C2",
    "PS-GKB": "C5",
    "FGK": "C3",
    "FLSQR-I": "C3",
    "FLSQR-R": "C3",
    "FLSQR-W": "C3",
    "IRW-FLSQR": "C3",
}

method_lines = {
    "GKS": "solid",
    "S-GKS": "solid",
    "resS-GKS": "dotted",
    "recS-GKS": "dashed",
    "PS-GKS": "solid",
    "resPS-GKS": "dotted",
    "recPS-GKS": "dashed",
    "PS-GKB": "solid",
    "FGK": "solid",
    "FLSQR-I": "solid",
    "FLSQR-R": "solid",
    "FLSQR-W": "solid",
    "IRW-FLSQR": "solid",
}



import numpy as np

CMAP = "gray"

from msolver import VanillaGKS, MM1GKS, MM2GKS, MM3GKS, IASGKS
from fsolver import MM2FGK, IASFGK
from pgkbsolver import MM3GKB, IASGKB
from pinv import SelectedWeightedNeumann2DPinvOperator


## Setup problem


In [ ]:
np.random.seed(0)
theta_max = 2*np.pi
#Tomo = Tomography (geom="fanflat", CommitCrime=True, theta_max=theta_max)
#Tomo = Tomography (geom="fanflat", CommitCrime=True)
Tomo = Tomography (geom="parallel", CommitCrime=False, theta_max=theta_max)
#views = 28 # until 4/6/25
views = 28 # until 4/6/25
noise_percent = 0.01 # percent of maximum of noiseless signal A x_true
N = 256

x_true = shepp_logan(N)
img_shape = x_true.shape
nx, ny = x_true.shape
(F , b_true , p , q , AforMatrixOperation ) = Tomo.gen_data( x_true , nx , ny , views )

noise_stdev = noise_percent*np.amax(b_true)
noise_var = noise_stdev**2
print(f"Noise level: {noise_percent}")
print(f"Noise stdev: {noise_stdev}")
print(f"Noise var: {noise_var}")


b_noisy = b_true.flatten() + noise_stdev*np.random.normal(size=b_true.flatten().shape)
b_noisy_shaped = b_noisy.reshape((p,q))
noisy_signal = b_noisy.flatten()
m, n = F.shape


# Define linear operator implementing A
_matvec_fn = lambda x: F.matvec(x)
_rmatvec_fn = lambda x: F.rmatvec(x)
A = _CustomLinearOperator( (m,n), _matvec_fn, _rmatvec_fn, device="cpu")

# Underdetermined rate:
underdetermined_rate = noisy_signal.size/(nx*ny)
print(f"Underdetermined rate: {underdetermined_rate}")


In [ ]:
np.savez_compressed(RESULTS / "tomography_nonconvex_data.npz", x_true=x_true,
                    b_true=b_true, b_noisy=b_noisy, b_noisy_shaped=b_noisy_shaped,
                    img_shape=np.asarray(img_shape), noise_var=noise_var,
                    noise_stdev=noise_stdev, views=views,
                    theta_max_requested=theta_max, angles=Tomo.theta,
                    source_origin=Tomo.source_origin, detector_origin=Tomo.detector_origin,
                    detector_spacing=Tomo.detector_pixel_size)


## Setup prior


In [ ]:
# Set method
Lmat = jlinops.build_neumann2d_sparse_matrix(img_shape)
L = jlinops.Neumann2D(img_shape)

K = np.ones(n)/np.linalg.norm(np.ones(n))
K = jlinops.MatrixLinearOperator(K)


def Lpinv_factory(w, *args, **kwargs):
    return SelectedWeightedNeumann2DPinvOperator(img_shape, weights=w, tol=1e-3, *args, **kwargs)


q = 0.5
mm2_weight_params = {
    "q": q,
}

mm_weight_params = {
    "q": q,
    "epsilon": 1e-4,
}

mmxx_weight_params = {
    "q": q,
    "epsilon": 1e-10,
}


The solver classes use fixed smoothing constants: `MM1GKS` uses 1, `MM2GKS` and `MM2FGK` use $10^{-2}$, and `MM3GKS` and `MM3GKB` use $10^{-3}$. These constants override the `epsilon` entries in the dictionaries above.


## Setup solvers


In [ ]:
# Build solvers
gks_solver = VanillaGKS(A, L, b_noisy, noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=K, img_shape=img_shape)
mm1_gks_solver = MM1GKS(A, L, b_noisy, noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=K, img_shape=img_shape)
mm2_gks_solver = MM2GKS(A, L, b_noisy, noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=K, img_shape=img_shape)
mm3_gks_solver = MM3GKS(A, L, b_noisy, noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=K, img_shape=img_shape)
mm3_gkb_solver = MM3GKB(A, L, b_noisy, noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=K, img_shape=img_shape)
mm2_fgk_solver = MM2FGK(A, L, b_noisy, noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=K, img_shape=img_shape)


In [ ]:
# Parameters
method = "dp"
n_its = 100
n_its_recres = 200

restart_dim = 40
rec_max_dim = 40
rec_min_dim = 25

# restart_dim = 35
# rec_max_dim = 35
# rec_min_dim = 20

# restart_dim = 60
# rec_max_dim = 60
# rec_min_dim = 40

ksparsity = None
lambdah_min = 1e-7
lambdah_max = 1e7
fgk_breakdown_tol = 1e-15


## Run all solvers


In [ ]:
all_v_results = []

# GKS
v_gks_data = gks_solver.solve(n_iter=n_its, init_dim=5, x0=None, lambdah0=1e0, lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                    regparam=method, x_true=x_true.flatten(), weight_params={},
                                    regparam_kwargs={"eta":1.01})

all_v_results.append( ( "V", "GKS", v_gks_data  ) )

save_solver_result(RESULTS / "tomography_nonconvex" / "v_gks.npz", v_gks_data)


### MM weights


In [ ]:
all_mm_results = []


In [ ]:
# S-GKS (2)
mm_sgks_data = mm2_gks_solver.solve(n_iter=n_its, init_dim=5, x0=v_gks_data["x"], lambdah0=v_gks_data["lambdah"], lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                    regparam=method, x_true=x_true.flatten(), weight_params=mm2_weight_params,
                                    regparam_kwargs={"eta":1.01})

all_mm_results.append( ( "MM", "S-GKS", mm_sgks_data  ) )

save_solver_result(RESULTS / "tomography_nonconvex" / "mm_sgks.npz", mm_sgks_data)


In [ ]:
# Recycled S-GKS (2)
mm_recsgks_data = mm2_gks_solver.solve(n_iter=n_its_recres, init_dim=5, x0=v_gks_data["x"], lambdah0=v_gks_data["lambdah"], lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                    regparam=method, x_true=x_true.flatten(), weight_params=mm2_weight_params,
                                    regparam_kwargs={"eta":1.01}, restart_dim=None, min_dim=rec_min_dim, max_dim=rec_max_dim)

all_mm_results.append( ( "MM", "S-GKS (recycled)", mm_recsgks_data  ) )

save_solver_result(RESULTS / "tomography_nonconvex" / "mm_recsgks.npz", mm_recsgks_data)


In [ ]:
# Restarted S-GKS (1)
mm_ressgks_data = mm1_gks_solver.solve(n_iter=n_its_recres, init_dim=5, x0=v_gks_data["x"], lambdah0=v_gks_data["lambdah"], lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                    regparam=method, x_true=x_true.flatten(), weight_params=mm2_weight_params,
                                    regparam_kwargs={"eta":1.01}, restart_dim=restart_dim)

all_mm_results.append( ( "MM", "S-GKS (restarted)", mm_ressgks_data  ) )

save_solver_result(RESULTS / "tomography_nonconvex" / "mm_ressgks.npz", mm_ressgks_data)


In [ ]:
# PS-GKS
mm_psgks_data = mm3_gks_solver.solve_with_priorconditioning_mod(n_iter=n_its, init_dim=5, x0=v_gks_data["x"], lambdah0=v_gks_data["lambdah"], lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                                 regparam=method, x_true=x_true.flatten(), weight_params=mm_weight_params,
                                                 regparam_kwargs={"eta":1.01})

all_mm_results.append( ( "MM", "PS-GKS", mm_psgks_data  ) )

save_solver_result(RESULTS / "tomography_nonconvex" / "mm_psgks.npz", mm_psgks_data)


In [ ]:
# Restarted PS-GKS
mm_respsgks_data = mm3_gks_solver.solve_with_priorconditioning_mod(n_iter=n_its_recres, init_dim=5, x0=v_gks_data["x"], lambdah0=v_gks_data["lambdah"], lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                                 regparam=method, x_true=x_true.flatten(), weight_params=mm_weight_params, restart_dim=restart_dim,
                                                 regparam_kwargs={"eta":1.01})

all_mm_results.append( ( "MM", "PS-GKS (restarted)", mm_respsgks_data  ) )

save_solver_result(RESULTS / "tomography_nonconvex" / "mm_respsgks.npz", mm_respsgks_data)


In [ ]:
# Recycled PS-GKS
mm_recpsgks_data = mm3_gks_solver.solve_with_priorconditioning_mod(n_iter=n_its_recres, init_dim=5, x0=v_gks_data["x"], lambdah0=v_gks_data["lambdah"], lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                                 regparam=method, x_true=x_true.flatten(), weight_params=mm_weight_params, min_dim=rec_min_dim, max_dim=rec_max_dim,
                                                 regparam_kwargs={"eta":1.01})

all_mm_results.append( ( "MM", "PS-GKS (recycled)", mm_recpsgks_data  ) )

save_solver_result(RESULTS / "tomography_nonconvex" / "mm_recpsgks.npz", mm_recpsgks_data)


In [ ]:
# PS-GKB solver
mm_gkb_data = mm3_gkb_solver.solve(n_iter=n_its, x0=None, lambdah0=None,
                                                 regparam=method, x_true=x_true.flatten(), ksparsity=None, weight_params=mm_weight_params,
                                                 lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                                #  regparam_kwargs={"eta":1.01},
                                                #  lambdah_min=1e-7, lambdah_max=1e7,
                                                #  init_dim=5
                                                 )
all_mm_results.append( ( "MM", "PS-GKB", mm_gkb_data  ) )

save_solver_result(RESULTS / "tomography_nonconvex" / "mm_gkb.npz", mm_gkb_data)


In [ ]:
# FGK solver (with 2, trying it out)
mm_fgk_data = mm2_fgk_solver.solve(n_iter=n_its, x0=None, lambdah0=None, breakdown_tol=fgk_breakdown_tol,
                                                 regparam=method, x_true=x_true.flatten(), ksparsity=None, weight_params=mmxx_weight_params, refresh_basis=False, lambdah_min=lambdah_min, lambdah_max=lambdah_max,
                                                #  regparam_kwargs={"eta":1.01},
                                                #  lambdah_min=1e-7, lambdah_max=1e7,
                                                #  init_dim=5
                                                 )

save_solver_result(RESULTS / "tomography_nonconvex" / "mm_fgk.npz", mm_fgk_data)


## Table 4


In [ ]:
# Refresh if needed
all_mm_results = [
    ( "MM", "S-GKS",  mm_sgks_data),
    ("MM", "resS-GKS", mm_ressgks_data),
    ("MM", "recS-GKS", mm_recsgks_data),
    ("MM", "PS-GKS", mm_psgks_data),
    ("MM", "resPS-GKS", mm_respsgks_data),
    ("MM", "recPS-GKS", mm_recpsgks_data),
    ("MM", "PS-GKB", mm_gkb_data),
    ("MM", "FGK", mm_fgk_data),
]


In [ ]:
v_df = pd.DataFrame({"Weights":[], "Method":[], "lambdah":[], "RRE":[], "SSIM": [], "HaarPSI":[], "Gini index": [], "Hoyer index": [], "A matvecs": [], "L matvecs": [], "WLpinv matvecs": [], "n_iters": []   })

for set in all_v_results:
        weight_str, method_label, method_data = set
        new_df = pd.DataFrame({"Weights":[weight_str], "Method":[method_label], "lambdah":[ method_data["lambda_history"][-1] ], "RRE":[ method_data["rre_history"][-1] ], "SSIM": [ method_data["ssim_history"][-1] ], "HaarPSI":[ method_data["haar_history"][-1] ], "Gini index": [ method_data["gini_indices"][-1] ], "Hoyer index": [ method_data["hoyer_indices"][-1] ] , "A matvecs": [ method_data["tot_matvec_A"] ], "L matvecs": [ method_data["tot_matvec_L"] ], "WLpinv matvecs": [ method_data["tot_matvec_WLpinv"] ], "n_iters": [ method_data["n_completed_iterations"] ]   })
        v_df = pd.concat([v_df, new_df], axis=0)


mm_df = pd.DataFrame({"Weights":[], "Method":[], "lambdah":[], "RRE":[], "SSIM": [], "Gini index": [], "Hoyer index": []})

for set in all_mm_results:
        weight_str, method_label, method_data = set
        # print(method_label)
        # print(method_data["rre_history"][-1])
        new_df = pd.DataFrame({"Weights":[weight_str], "Method":[method_label], "lambdah":[ method_data["lambda_history"][-1] ], "RRE":[ method_data["rre_history"][-1] ], "SSIM": [ method_data["ssim_history"][-1] ], "HaarPSI":[ method_data["haar_history"][-1] ], "Gini index": [ method_data["gini_indices"][-1] ], "Hoyer index": [ method_data["hoyer_indices"][-1] ],  "A matvecs": [ method_data["tot_matvec_A"] ], "L matvecs": [ method_data["tot_matvec_L"] ], "WLpinv matvecs": [ method_data["tot_matvec_WLpinv"] ], "n_iters": [ method_data["n_completed_iterations"] ]   })
        mm_df = pd.concat([mm_df, new_df], axis=0)


In [ ]:
# all_df = pd.concat([v_df, mm_df, ias_df])
# all_df = all_df.rename(columns={'lambdah': '$\lambda$'})
# all_df

all_df = pd.concat([v_df, mm_df])
all_df = all_df[["Weights", "Method", "lambdah", "n_iters", "RRE", "SSIM", "Gini index", "A matvecs", "L matvecs", "WLpinv matvecs"]]
all_df = all_df.rename(columns={'lambdah': '$\lambda$', "n_iters":"$n_{\\textrm{iter}}$", "A matvecs":"$n_{\mathbf{A}}$", "L matvecs":"$n_{\mathbf{L}}$", "WLpinv matvecs":"$n_{\mathbf{ L }_k^\dagger }$"  })

all_df.to_csv(RESULTS / "table_04.csv", index=False)
save_table_png(all_df, RESULTS / "table_04.png", lambda_digits=2)
all_df


In [ ]:
def scientific_to_latex(sci_str, digits=1):
    # Split the number into base and exponent parts
    # Table 4 uses two significant digits for the regularization parameter.
    sci_str = f"{sci_str:.{digits - 1}e}"
    base, exponent = sci_str.split('e')
    
    # Convert base to integer (since the decimal places can be ignored)
    base = str(int(float(base))) if digits == 1 else base
    
    # Prepare the exponent part, removing the leading "+" if it's a positive exponent
    exponent = exponent.lstrip("+")
    
    # Format the result as a LaTeX string
    latex_str = f"${base}\\cdot10^{{{int(exponent)}}}$"
    
    return latex_str


In [ ]:
print( all_df.to_latex(index=False, formatters={'$\lambda$': lambda s: scientific_to_latex(s, digits=2), 
                                                "RRE": "${:.3f}$", "SSIM": "${:.3f}$", 
                                                  "Gini index": "${:.3f}$", "Hoyer index": "${:.3f}$", 
                                                  "$n_{\\textrm{iter}}$": "${:.0f}$",
                                                  "$n_{\mathbf{A}}$": "${:.0f}$", "$n_{\mathbf{L}}$": "${:.0f}$", "$n_{\mathbf{ L }_k^\dagger }$": "${:.0f}$"
                                                    }) )


In [ ]:
save_raw_results(RESULTS / "tomography_nonconvex_raw.npz", all_v_results + all_mm_results)


The CSV files contain the values from this run. The `.npz` files contain numeric arrays only and can be opened with `np.load(..., allow_pickle=False)`. Individual solver results are saved as each method finishes.
